# Глава 5. KNN: алгоритм ближайших соседей

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить принцип работы KNN своими словами, построить классификатор KNN средствами scikit-learn и описать, как параметр K влияет на поведение модели.

В первой фазе мы разглядывали данные. Теперь научим машину предсказывать. Начнём с самого понятного алгоритма во всём машинном обучении — метода k ближайших соседей, коротко KNN (k-Nearest Neighbors). Если ты поймёшь его, ты поймёшь саму идею предсказания по данным.

**Аналогия: скажи мне, кто твой сосед  💡**

KNN — это «ленивый ученик». Он ничему не учится заранее и не выводит никаких правил. Когда нужно классифицировать новый объект, он просто смотрит: а на кого этот объект похож среди тех, кого я уже знаю? Находит несколько самых близких «соседей» и повторяет их ответ. Скажи мне, кто твои соседи, — и я скажу, кто ты.

## Как это работает по шагам

Представь точки на плоскости: синие — «сдал экзамен», красные — «не сдал». Их положение задают два признака (например, часы учёбы и часы сна). Появляется новая точка без цвета — новый ученик, для которого мы хотим предсказать результат. Что делает KNN?

Измеряет расстояние от новой точки до всех известных точек. Ближе точка — больше похож объект.

Выбирает k ближайших (например, k = 5 — пять самых близких точек).

Смотрит, какого цвета большинство из этих пяти соседей.

Присваивает новой точке цвет большинства. Голосование окончено, предсказание готово.

Заметь: KNN не строит формул и не «понимает» данные. Он лишь измеряет похожесть и голосует. Именно поэтому его называют ленивым — вся «работа» откладывается до момента предсказания.

## Параметр K — сколько соседей спрашивать

K — это главная настройка алгоритма (её называют гиперпараметром — значением, которое задаёт человек, а не выучивает модель). K задаёт, скольких соседей мы опрашиваем. И выбор K сильно меняет поведение модели.

Значение K

Что происходит

Риск

K = 1

Смотрим только на одного ближайшего соседа

Модель «шумит», реагирует на случайные точки и выбросы

K средний (5–15)

Голосует небольшая устойчивая группа

Обычно оптимально

K очень большой

Голосует почти весь датасет

Модель «размывается», теряет детали, всё сводит к большинству

**Полезно знать  ℹ️**

K обычно берут нечётным (3, 5, 7…) — чтобы при голосовании двух классов не случилось ничьей. Как правильно и честно подбирать K, не обманывая себя, мы разберём в следующей главе — это отдельный важный навык.

## Живая игра без кода

Прежде чем писать код, почувствуй алгоритм телом. Расставьте по классу учеников двух «типов» — например, любителей спорта в один угол, любителей музыки в другой. Пусть новый ученик встанет где-то посередине и посмотрит на трёх ближайших к себе. Кого больше рядом — к той группе KNN его и отнесёт. Поменяйте K с 3 на 7 — заметьте, как ответ может измениться. Вы только что стали живым алгоритмом ближайших соседей.

## Первый классификатор в коде

Соберём KNN на данных о пингвинах Палмера: предскажем вид пингвина по длине его клюва и массе тела. Это наш первый настоящий обученный классификатор.

In [ ]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

# Загружаем данные и убираем строки с пропусками (пока просто так)
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url).dropna(subset=["bill_length_mm", "body_mass_g", "species"])

# X — признаки (по чему предсказываем), y — цель (что предсказываем)
X = df[["bill_length_mm", "body_mass_g"]]
y = df["species"]

# Создаём модель: спрашиваем 5 ближайших соседей
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X, y)          # "обучаем" — KNN просто запоминает точки

# Предсказываем для нового пингвина: клюв 45 мм, масса 4000 г
print(model.predict([[45, 4000]]))   # -> вид пингвина

_Ожидаемый результат: название вида (например, ['Adelie']) — прогноз для выдуманного пингвина._

Обрати внимание на два ключевых метода, которые встретятся в каждой модели scikit-learn: .fit() — обучение (модель запоминает данные) и .predict() — предсказание для новых объектов. Этот интерфейс единый для всех алгоритмов sklearn, поэтому, освоив его один раз, ты сможешь менять модели одной строкой.

**Внимание, частая ошибка  ⚠️**

KNN измеряет расстояние — значит, ему критически важен масштаб признаков. Длина клюва (~33–60 мм) и масса тела (~2700–6300 г) лежат в разных диапазонах, и масса «перевесит» длину клюва просто потому, что числа крупнее, а не потому, что она важнее. Это искажает расстояния и портит модель. Лечится масштабированием признаков — к нему мы вернёмся в главе 10, когда соберём Pipeline.

## Задания

Базовый уровень

Запусти код KNN из главы. Затем попробуй разные значения n_neighbors (1, 3, 5, 15) и посмотри, меняется ли предсказание для пингвина [45, 4000]. Запиши наблюдения.

Со звёздочкой  ⭐

Добавь к признакам ещё один столбец — flipper_length_mm (длина ласта). Изменилось ли предсказание? Подумай и запиши: почему добавление осмысленного признака может улучшить модель, а бессмысленного (например, случайного числа) — ухудшить?

Вопросы для обсуждения  💬

Почему KNN называют «ленивым» алгоритмом? В чём его лень — и это плюс или минус?

Что произойдёт с моделью, если задать K = 1? А если K равно числу всех объектов?

Придумай реальную задачу, где принцип «скажи, кто твои соседи» работает хорошо, и задачу, где он подведёт.

**Проверь себя  ℹ️**

1. Опиши работу KNN в четыре шага.

2. Почему K обычно берут нечётным?

3. Почему KNN чувствителен к масштабу признаков?